In [12]:
from langchain_postgres import PGVector
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv()

True

In [17]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")
RESUME_PATH = r"C:\Users\rolex\OneDrive\Desktop\RAG\data\sample.pdf"

In [18]:
loader = PyMuPDFLoader(RESUME_PATH)

document = loader.load()

len(document),document[0].page_content

(1,
 'Durga Prasad Kota\nAI/ML Engineer | Python Backend Developer\n7286945898\ndurgaprasad04289@gmail.com\nPortfolio\nLinkedIn\nGitHub\nEluru, India\nPROFESSIONAL SUMMARY\nAI/ML Engineer and Python Backend Developer with hands-on experience in Python, Django, FastAPI, and REST APIs.\nProficient in developing scalable backend systems, ML-powered APIs, and predictive models, as well as full-stack applications\nusing React.js and PostgreSQL. Skilled in integrating AI/ML solutions into production-ready web applications.\nEDUCATIONS\nRamachandra College of Engineering\nB.Tech in Computer Science Engineering\n2024 - 2028\nCGPA : 8.9\nVenkateswara Junior College\nSenior Secondary School Certificate (SSCE)\n2022 - 2024\nCGPA : 95.6%\nPROJECTS\n|\nSalary Prediction\nGithub\nWebsite\nIndividual Project\nDeveloped a regression-based ML system to predict employee salaries using Years of Experience and Age, incorporating data\npreprocessing and feature analysis.\nTrained and evaluated a Linear Reg

In [19]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(document)

chunks

[Document(metadata={'producer': 'Skia/PDF m153', 'creator': 'Chromium', 'creationdate': '2026-09-25T05:00:36+00:00', 'source': 'C:\\Users\\rolex\\OneDrive\\Desktop\\RAG\\data\\sample.pdf', 'file_path': 'C:\\Users\\rolex\\OneDrive\\Desktop\\RAG\\data\\sample.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Durga_Prasad_Kota_Resume_CZ.pdf', 'author': '', 'subject': '', 'keywords': 'embedded_resume_metadata', 'moddate': '2026-09-25T05:00:36+00:00', 'trapped': '', 'modDate': "D:20260925050036+00'00'", 'creationDate': "D:20260925050036+00'00'", 'page': 0}, page_content='Durga Prasad Kota\nAI/ML Engineer | Python Backend Developer\n7286945898\ndurgaprasad04289@gmail.com\nPortfolio\nLinkedIn\nGitHub\nEluru, India\nPROFESSIONAL SUMMARY\nAI/ML Engineer and Python Backend Developer with hands-on experience in Python, Django, FastAPI, and REST APIs.\nProficient in developing scalable backend systems, ML-powered APIs, and predictive models, as well as full-stack applications'),
 Document(met

In [ ]:
llm = init_chat_model(
    model="qwen/qwen3.8-27b",
    model_provider="groq",
    api_key=GROQ_API_KEY,
    
)

In [25]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    api_key=GEMINI_API_KEY
)

In [26]:
test_query = embedding_model.embed_query("Who is the resume holder ")

print(len(test_query))

3072


In [29]:
import psycopg2

conn = psycopg2.connect(DATABASE_URL)

cursor = conn.cursor()

cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")

conn.commit()

print("pgvector extension is working!")

cursor.close()
conn.close()

pgvector extension is working!


In [34]:
vector_database = PGVector(
    embeddings=embedding_model,
    collection_name="resume_documents",
    connection=DATABASE_URL,
    use_jsonb=True
)

print("PGVector connected successfully!")

PGVector connected successfully!


In [52]:
# ids = vector_database.add_documents(chunks)

# print("Stored:", len(ids))
# print("PGVector connected + embeddings stored successfully!")

In [44]:
query = "What backend technologies does Durga use?"

retriver = vector_database.as_retriever(
    search_kwargs = {'k':3}
)

results = retriver.invoke(query)

In [45]:
results

[Document(id='7856171b-41d0-43a1-a925-3eed0fe163dd', metadata={'page': 0, 'title': 'Durga_Prasad_Kota_Resume_CZ.pdf', 'author': '', 'format': 'PDF 1.4', 'source': 'C:\\Users\\rolex\\OneDrive\\Desktop\\RAG\\data\\sample.pdf', 'creator': 'Chromium', 'modDate': "D:20260925050036+00'00'", 'moddate': '2026-09-25T05:00:36+00:00', 'subject': '', 'trapped': '', 'keywords': 'embedded_resume_metadata', 'producer': 'Skia/PDF m153', 'file_path': 'C:\\Users\\rolex\\OneDrive\\Desktop\\RAG\\data\\sample.pdf', 'total_pages': 1, 'creationDate': "D:20260925050036+00'00'", 'creationdate': '2026-09-25T05:00:36+00:00'}, page_content='Durga Prasad Kota\nAI/ML Engineer | Python Backend Developer\n7286945898\ndurgaprasad04289@gmail.com\nPortfolio\nLinkedIn\nGitHub\nEluru, India\nPROFESSIONAL SUMMARY\nAI/ML Engineer and Python Backend Developer with hands-on experience in Python, Django, FastAPI, and REST APIs.\nProficient in developing scalable backend systems, ML-powered APIs, and predictive models, as well 

In [55]:
for i, doc in enumerate(results, 1):
    if i==1:
        print(f"\n--- Chunk {i} ---")
        print(doc.page_content)


--- Chunk 1 ---
Durga Prasad Kota
AI/ML Engineer | Python Backend Developer
7286945898
durgaprasad04289@gmail.com
Portfolio
LinkedIn
GitHub
Eluru, India
PROFESSIONAL SUMMARY
AI/ML Engineer and Python Backend Developer with hands-on experience in Python, Django, FastAPI, and REST APIs.
Proficient in developing scalable backend systems, ML-powered APIs, and predictive models, as well as full-stack applications
